## Reading in and combining files

In [56]:
import pandas as pd
import numpy as np

### Read in both files

In [57]:
columbia_df = pd.read_csv('../data/Columbia_filtered.csv')
hcmst_df = pd.read_csv('../data/hCMST_filtered.csv')

In [58]:
hcmst_df.columns

Index(['Subject age', 'Partner age', 'Subject race', 'Partner race',
       'Interracial relationship', 'Hispanic origin of partner',
       'Subject Hispanic origin', 'Subject gender', 'Partner gender',
       'Attraction', 'Subject education', 'Partner education', 'Region',
       'Subject's employment status', 'Earn more money?', 'Same high school?',
       'Same college?', 'Grew up same city?', 'Living Together',
       'Married at baseline', 'Quality of Relationship?', 'Same_Age_Bracket',
       'match'],
      dtype='str')

### Columns in Both Dataset

In [59]:
target_columns_dict = [
    "Subject age",
    "Partner age",
    "Subject race",
    "Partner race",
    "Subject gender",
    "Partner gender",
    "match",
    "Same_Age_Bracket",
    'Interracial relationship' 
]

In [60]:
columbia_df = columbia_df[target_columns_dict]  
hcmst_df = hcmst_df[target_columns_dict]

In [61]:
print(columbia_df.shape)
columbia_df.head()

(7994, 9)


,Subject age,Partner age,Subject race,Partner race,Subject gender,Partner gender,match,Same_Age_Bracket,Interracial relationship
0,21,21,ASIAN,WHITE,0,1,0,1,1
1,21,21,ASIAN,WHITE,0,1,0,1,1
2,21,21,ASIAN,ASIAN,0,1,1,1,0
3,21,21,ASIAN,WHITE,0,1,1,1,1
4,21,21,ASIAN,HISPANIC,0,1,1,1,1


In [62]:
print(hcmst_df.shape)
hcmst_df.head()

(3385, 9)


,Subject age,Partner age,Subject race,Partner race,Subject gender,Partner gender,match,Same_Age_Bracket,Interracial relationship
0,41,41,NATIVE AMERICAN,WHITE,1,0,1,1,1
1,11,71,WHITE,WHITE,1,0,1,0,0
2,21,41,WHITE,WHITE,0,1,1,0,0
3,21,51,WHITE,ASIAN,0,1,1,0,1
4,41,31,WHITE,WHITE,0,1,1,0,0


In [63]:
binary_cols = [ 
    "Subject gender", 
    "Partner gender", 
    'Interracial relationship'
]

# Need to One-Hot Encoding)
nominal_cols = ['Subject race', 
                'Partner race', 
                ]

# leave as numeric, scale if needed
continuous_cols = [ ]

# leave as numeric, scale if needed
ordinal_cols = [
    'Subject age', 
    'Partner age'
]

## COMBINE DATASET

In [64]:
# ADD A COLUMN SO WE KNOW WHICH ROW CAME FROM WHICH DATASET
columbia_df['Dataset_Source'] = 'Columbia'
hcmst_df['Dataset_Source'] = 'hCMST'

# 2. Combine them
combined_df = pd.concat([columbia_df, hcmst_df], ignore_index=True)

In [65]:
combined_df.columns

Index(['Subject age', 'Partner age', 'Subject race', 'Partner race',
       'Subject gender', 'Partner gender', 'match', 'Same_Age_Bracket',
       'Interracial relationship', 'Dataset_Source'],
      dtype='str')

## Prepare

### Drop Rows with Missing values

In [66]:
combined_df = combined_df.dropna().reset_index(drop=True)
print(combined_df.isna().sum())


Subject age                 0
Partner age                 0
Subject race                0
Partner race                0
Subject gender              0
Partner gender              0
match                       0
Same_Age_Bracket            0
Interracial relationship    0
Dataset_Source              0
dtype: int64


### Encode Catagorical Variables

In [67]:
combined_df = pd.get_dummies(combined_df, columns=nominal_cols, dtype=int)

### Standardize Age

Using the MinMaxScaler for our Age variables because they are binned (i.e. 21-30). This is to prevent distance bias when we do our clustering. 

MinMax = Compress data into a fixed range (0 to 1)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

for col in ordinal_cols:
    combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

combined_df = combined_df.dropna(subset=ordinal_cols)

scaler = MinMaxScaler()
combined_df[ordinal_cols] = scaler.fit_transform(combined_df[ordinal_cols])


In [71]:
print(combined_df.shape)
combined_df.head()

(11289, 22)


,Subject age,Partner age,Subject gender,Partner gender,match,Same_Age_Bracket,Interracial relationship,Dataset_Source,Subject race_ASIAN,Subject race_BLACK,...,Subject race_NATIVE AMERICAN,Subject race_OTHER,Subject race_WHITE,Partner race_ASIAN,Partner race_BLACK,Partner race_HISPANIC,Partner race_LATINO,Partner race_NATIVE AMERICAN,Partner race_OTHER,Partner race_WHITE
0,0.344262,0.305556,0,1,0,1,1,Columbia,1,0,...,0,0,0,0,0,0,0,0,0,1
1,0.344262,0.305556,0,1,0,1,1,Columbia,1,0,...,0,0,0,0,0,0,0,0,0,1
2,0.344262,0.305556,0,1,1,1,0,Columbia,1,0,...,0,0,0,1,0,0,0,0,0,0
3,0.344262,0.305556,0,1,1,1,1,Columbia,1,0,...,0,0,0,0,0,0,0,0,0,1
4,0.344262,0.305556,0,1,1,1,1,Columbia,1,0,...,0,0,0,0,0,1,0,0,0,0


In [72]:
combined_df.to_csv('../data/combined_dataset_matching_features.csv', index=False)